In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import hstack, kron, eye, csc_matrix, block_diag


def repetition_code(n):
    """
    Parity check matrix of a repetition code with length n.
    """
    row_ind, col_ind = zip(*((i, j) for i in range(n) for j in (i, (i+1)%n)))
    data = np.ones(2*n, dtype=np.uint8)
    return csc_matrix((data, (row_ind, col_ind)))


def toric_code_x_stabilisers(L):
    """
    Sparse check matrix for the X stabilisers of a toric code with
    lattice size L, constructed as the hypergraph product of
    two repetition codes.
    """
    Hr = repetition_code(L)
    H = hstack(
            [kron(Hr, eye(Hr.shape[1])), kron(eye(Hr.shape[0]), Hr.T)],
            dtype=np.uint8
        )
    H.data = H.data % 2
    H.eliminate_zeros()
    return csc_matrix(H)

In [2]:
def toric_code_x_logicals(L):
    """
    Sparse binary matrix with each row corresponding to an X logical operator
    of a toric code with lattice size L. Constructed from the
    homology groups of the repetition codes using the Kunneth
    theorem.
    """
    H1 = csc_matrix(([1], ([0],[0])), shape=(1,L), dtype=np.uint8)
    H0 = csc_matrix(np.ones((1, L), dtype=np.uint8))
    x_logicals = block_diag([kron(H1, H0), kron(H0, H1)])
    x_logicals.data = x_logicals.data % 2
    x_logicals.eliminate_zeros()
    return csc_matrix(x_logicals)

In [3]:
toric_code_x_logicals(2).toarray()

array([[1, 1, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 1, 0, 1, 0]], dtype=uint8)

In [4]:
H = toric_code_x_stabilisers(2)

My own tests with my notation, I am understanding things or not?

In [5]:
toric_code_2_check = np.array([
    [1, 0, 1, 1, 1, 0, 0, 0],
    [0, 1, 1, 1, 0, 1, 0, 0],
    [1, 0, 0, 0, 1, 0, 1, 1],
    [0, 1, 0, 0, 0, 1, 1, 1]
])

noise = np.array([1, 0, 0, 0, 0, 1, 0, 0])

syndrome = (toric_code_2_check @ noise) % 2
syndrome

array([1, 1, 1, 1])

In [6]:
from pymatching import Matching

matching = Matching(toric_code_2_check)

In [7]:
prediction = matching.decode(syndrome)
prediction

array([1, 1, 0, 0, 0, 0, 0, 0], dtype=uint8)

In [8]:
from utils.postselection import pauli_to_str, diagonal_operators_check

def chain_check_matrix(postselection_ops):
    all_diagonal, basis = diagonal_operators_check(postselection_ops)
    if not all_diagonal:
        raise ValueError("All postselection operators must be diagonal")
    check_matrix = np.array([[int(str(opel) == basis) for opel in operator] for operator in postselection_ops])
    return check_matrix

In [9]:
from qiskit.quantum_info import Pauli, PauliList

pops = PauliList([Pauli("ZZIII"), Pauli("IZZZI"), Pauli("IIIZZ")])

chain_check = chain_check_matrix(pops)
chain_check

array([[0, 0, 0, 1, 1],
       [0, 1, 1, 1, 0],
       [1, 1, 0, 0, 0]])

In [10]:
from pymatching import Matching

matching = Matching(chain_check)

all_states = np.array([list(f"{i:05b}") for i in range(2**5)], dtype=int)
clean_states = []

for state in all_states:
    syndrome = (chain_check @ state) % 2
    predicted_flips = matching.decode(syndrome)
    final_state = (state + predicted_flips) % 2
    if len(clean_states) == 0 or not np.any(np.all(np.equal(final_state, clean_states), axis=1)):
        clean_states.append(final_state)

clean_states

[array([0, 0, 0, 0, 0]),
 array([0, 0, 1, 1, 1]),
 array([1, 1, 0, 1, 1]),
 array([1, 1, 1, 0, 0])]

In [11]:
from pymatching import Matching
from z2chain.operators import postselection_operators

L = 5
nqubits = 2*L-1

chain_check = chain_check_matrix(postselection_operators(L))
matching = Matching(chain_check)

all_states = np.array([list(f"{i:0{nqubits}b}") for i in range(2**nqubits)], dtype=int)
clean_states = []

for state in all_states:
    syndrome = (chain_check @ state) % 2
    predicted_flips = matching.decode(syndrome)
    final_state = (state + predicted_flips) % 2
    if len(clean_states) == 0 or not np.any(np.all(np.equal(final_state, clean_states), axis=1)):
        clean_states.append(final_state)

clean_states

[array([0, 0, 0, 0, 0, 0, 0, 0, 0]),
 array([0, 0, 0, 0, 0, 0, 1, 1, 1]),
 array([0, 0, 0, 0, 1, 1, 0, 1, 1]),
 array([0, 0, 0, 0, 1, 1, 1, 0, 0]),
 array([0, 0, 1, 1, 0, 1, 0, 1, 1]),
 array([0, 0, 1, 1, 0, 1, 1, 0, 0]),
 array([0, 0, 1, 1, 1, 0, 0, 0, 0]),
 array([0, 0, 1, 1, 1, 0, 1, 1, 1]),
 array([1, 1, 0, 1, 0, 1, 0, 1, 1]),
 array([1, 1, 0, 1, 0, 1, 1, 0, 0]),
 array([1, 1, 0, 1, 1, 0, 0, 0, 0]),
 array([1, 1, 0, 1, 1, 0, 1, 1, 1]),
 array([1, 1, 1, 0, 0, 0, 0, 0, 0]),
 array([1, 1, 1, 0, 0, 0, 1, 1, 1]),
 array([1, 1, 1, 0, 1, 1, 0, 1, 1]),
 array([1, 1, 1, 0, 1, 1, 1, 0, 0])]

In [12]:
from pymatching import Matching
from z2chain.operators import postselection_operators

L = 5
nqubits = 2*L-1

chain_check = chain_check_matrix(postselection_operators(L))
matching = Matching(chain_check)

state = [0, 1, 0, 1, 0, 1, 0, 1, 0]
syndrome = (chain_check @ state) % 2
predicted_flips = matching.decode(syndrome)
final_state = (state + predicted_flips) % 2
final_state

array([1, 1, 0, 1, 0, 1, 0, 1, 1])

In [84]:
from pymatching import Matching
from z2chain.operators import postselection_operators

L = 11
nqubits = 2*L-1

chain_check = chain_check_matrix(postselection_operators(L))
matching = Matching(chain_check)

all_states = np.array([list(f"{i:0{nqubits}b}") for i in range(2**nqubits)], dtype=int)
clean_states_receipts = {}

for state in all_states:
    syndrome = (chain_check @ state) % 2
    predicted_flips = matching.decode(syndrome)
    nflips = predicted_flips.sum()
    final_state = (state + predicted_flips) % 2
    final_state_str = "".join(final_state.astype(str).tolist())
    if clean_states_receipts.get(final_state_str, None) is not None:
        clean_states_receipts[final_state_str][nflips] += 1
    else:
        clean_states_receipts[final_state_str] = {i: 1 if nflips == i else 0 for i in range(L)}
clean_states_receipts

{'000000000000000000000': {0: 1,
  1: 21,
  2: 162,
  3: 560,
  4: 840,
  5: 432,
  6: 32,
  7: 0,
  8: 0,
  9: 0,
  10: 0},
 '000000000000000000111': {0: 1,
  1: 21,
  2: 162,
  3: 560,
  4: 840,
  5: 432,
  6: 32,
  7: 0,
  8: 0,
  9: 0,
  10: 0},
 '000000000000000011011': {0: 1,
  1: 21,
  2: 162,
  3: 560,
  4: 840,
  5: 432,
  6: 32,
  7: 0,
  8: 0,
  9: 0,
  10: 0},
 '000000000000000011100': {0: 1,
  1: 21,
  2: 162,
  3: 560,
  4: 840,
  5: 432,
  6: 32,
  7: 0,
  8: 0,
  9: 0,
  10: 0},
 '000000000000001101011': {0: 1,
  1: 21,
  2: 162,
  3: 560,
  4: 840,
  5: 432,
  6: 32,
  7: 0,
  8: 0,
  9: 0,
  10: 0},
 '000000000000001101100': {0: 1,
  1: 21,
  2: 162,
  3: 560,
  4: 840,
  5: 432,
  6: 32,
  7: 0,
  8: 0,
  9: 0,
  10: 0},
 '000000000000001110000': {0: 1,
  1: 21,
  2: 162,
  3: 560,
  4: 840,
  5: 432,
  6: 32,
  7: 0,
  8: 0,
  9: 0,
  10: 0},
 '000000000000001110111': {0: 1,
  1: 21,
  2: 162,
  3: 560,
  4: 840,
  5: 432,
  6: 32,
  7: 0,
  8: 0,
  9: 0,
  10: 0},


In [22]:
(2*L-1)-(L-1)

5

In [18]:
2**5

32